## 1. Imports et configuration

Cet import est à executer si jamais on lance le notebook via google colab. Déposer au préalable le dossier food-11.zip dans son drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/food-11.zip" -d /content/
!ls -la /content/

On lance ces différents imports et configuration pour pytorch:

- `torch:` pour la manipulation de tenseurs, ainsi que pour le calcul différentiel.
- `nn:` contient les élements pour l'architecture du réseau (couches de calcul qui contiennent poids et biais ajustés pendant l'entrainement (linear ou convolution), fonctions d'activations non linéaires entre couches, fonctions de perte qui quantifient l'écart entre les prédicitons et la vraie étiquette) ainsi que la tstructure du modèle (nn.Module)
- `optim:` contient les algo d'optimisation (ici adam) qui ajustent poids du réseau après chaque calcul d'erreur.
- `dataloader:` permet de découper le dataset en batch et de les mélanger (shuffle) ainsi que les charger parallèlement.
- `random_split` pour créer un subset de train et de validation
- `dataset:` permet d'automatiquement lire des dossiers classés par étiquette
- `transforms:` permet de redimensionner images, les convertir en tenseurs et appliquer variation aléatoire pour éviter l'overfitting
- `models:` est utilisé pour importer réseau préentrainé (resnet50) 

Ensuite, en fonction de l'appareil, sélection du meilleur GPU, mps pour appple, cuda pour nvidia.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Device :", device)


Device : mps


## 2. Chemins et paramètres

`ImageFolder` déduit automatiquement les labels des images à partir des sous-dossiers. On configure les chemins des fichiers, ainsi que divers paramètres (epochs, learning rate, taille des batchs, des images...)

In [ ]:
DATA_DIR = Path("food-11")
TRAIN_DIR = DATA_DIR / "train"
MODEL_PATH = Path("model.pth")

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 12
LEARNING_RATE = 1e-3

print("Classes disponibles :", sorted(path.name for path in TRAIN_DIR.iterdir() if path.is_dir()))


Classes disponibles : ['bread', 'dairy', 'dessert', 'egg', 'fried', 'meat', 'noodles-pasta', 'rice', 'seafood', 'soup', 'vegetable-fruit']


## 3. Transformations et DataLoaders

Les images sont redimensionnées avec la valeur IMAGE_SIZE définie, normalisées, on applique une transformation aléatoire en rotation pour éviter l'overfitting.

- ImageFolder parcourt les dossiers et étiquette chaque image selon le nom du répertoire (pytorch crée un dictionnaire class_to_idx qui relie le num d'étiquette à une catégorie) (trois matrices de 128×128 superposées : une matrice Rouge, une Verte, une Bleue, étiquette)

- On sépare les images de manière aléatoire, 85% servent à l'entrainement et 15% à la validation.

- Le DataLoader crée des lots de taille batch size (indépendamment de la classe d'origine) et les envoie au modèle pendant l'entrainement

In [ ]:
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    normalize,
])

dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
n_train = int(0.85 * len(dataset))

train_dataset, test_dataset = random_split(
    dataset, [n_train, len(dataset) - n_train]
)

class_names = dataset.classes
num_classes = len(class_names)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train : {len(train_dataset)} images, {len(train_loader)} lots")
print(f"Test  : {len(test_dataset)} images, {len(test_loader)} lots")


## 4. Définir le modèle

- D'abord, on charge le modèle resnet 50 préentrainé, qui initialise des poids cohérents. 

- avec self.backbone.fc, on remplace la couche finale par un bloc nn.Sequential qui normalise les caractéristiques d sortie, et effectue une projection vers chaque classe cible. 

- Ensuite, on instancie le modèle et on transfère les poids vers le GPU

- On applique un `label_smoothing` pour que je modèle ne soit pas trop sur de lui

- On ajuste la vitesse d'apprentissage de l'optimiseur adamW, le learning_rate est 10 fois plus faible pour les couches préentrainées et la nouvelle couche de sortie à pleine vitesse. On réduit la vitesse d'apprentissage au cours du temps `CosineAnnealingLR`

In [ ]:
class Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

model = Model(num_classes).to(device)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

backbone_params = [p for name, p in model.backbone.named_parameters() if not name.startswith("fc.")]
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": LEARNING_RATE * 0.1},
    {"params": model.backbone.fc.parameters(), "lr": LEARNING_RATE},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print(model)


## 5. Fonctions d'entraînement

On initialise les compteurs à 0. On parcourt les données par batch. Il réinitialise les gradients des calculs précédents. Il effectue une prediction avec `model(X)`. Calcule l'écart avec la loss function, at calcule l'ajustement de chaque paramètre avec `backward` pour la réduire. Ensuite avec `step` il met a jour les paramètres du modèle. 

In [9]:
def train_one_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        predictions = model(X)
        loss = loss_fn(predictions, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        correct += (predictions.argmax(dim=1) == y).sum().item()
        total += X.size(0)

    return total_loss / total, correct / total

## 6. Fonction d'évaluation

La fonction evaluate mesure les performances sur le jeu de validation. Aucun apprentissage (pas de `backward`, `step`)

In [10]:
def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = (model(X) + model(X.flip(-1))) / 2
            loss = loss_fn(predictions, y)
            total_loss += loss.item() * X.size(0)
            correct += (predictions.argmax(dim=1) == y).sum().item()
            total += X.size(0)

    return total_loss / total, correct / total

## 7. Entraîner le modèle

Pour chaqu époch, il utilise la fonction train_one_epoch, puis evaluate. Le `learning_rate` est ajustée à chaque époch pour mieux converger. Si le modèle a un meilleur résultat sur le set de validation, il est sauvegardé. On sauvegarde l'évolution de la précision et des pertes pour tracer un graphique matplotlib plus tard.

In [ ]:
import copy

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
best_acc = 0.0
best_state_dict = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    scheduler.step()

    if test_acc > best_acc:
        best_acc = test_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
        f"test_loss={test_loss:.4f} | test_acc={test_acc:.3f} | "
        f"lr={scheduler.get_last_lr()[0]:.6f}"
    )

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print(f"Meilleur modèle restauré (test_acc={best_acc:.3f})")


## 8. Visualiser les courbes d'apprentissage

On trace le loss du train et test en fonction des epoch. Ensuite on trace la précision train/test en fonction des époch pour voir l'évolution des performances

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['test_loss'], label='Test')
axes[0].set_title('Perte')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['test_acc'], label='Test')
axes[1].set_title('Exactitude')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

### Matrice de confusion

On évalue le modèle sur le jeu de validation (`test_loader`) pour visualiser la matrice de confusion par classe.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        logits = (model(X) + model(X.flip(-1))) / 2
        preds = logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(y.tolist())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=45
)
ax.set_title("Matrice de confusion (validation)")
plt.tight_layout()
plt.show()

## 9. Sauvegarder et recharger le modèle

On sauvegarde l'état complet du modèle entrainé dans checkpoint. 

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'image_size': IMAGE_SIZE,
}, MODEL_PATH)
print(f'Modèle sauvegardé dans : {MODEL_PATH}')

On recharge le modèle pour faire des prédictions sans avoir à le réentrainer

In [ ]:

checkpoint = torch.load(MODEL_PATH, map_location=device)
loaded_model = Model(len(checkpoint['class_names'])).to(device)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()
print('Modèle rechargé avec succès.')

## 10. Générer le fichier JSON de prédictions (`food-11`)

On parcourt et trie toutes les images sans étiquettes (dossier food-11/test). On crée alors des batch. On utilis le modèle pour prédire classe (en utilisant la moyenne de l'image originale et en mirroir pour une meilleure fiabilité). On associe alors le numéro de l'image à la classe ("bread" etc)

In [ ]:
import json
from PIL import Image

test_dir = Path("food-11/test")

output_json_path = Path("predictions.json")

# Tri des images numériquement selon leur identifiant (0.jpg, 1.jpg, ...)
image_paths = sorted(
    [p for p in test_dir.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")],
    key=lambda p: int(p.stem)
)

predictions_dict = {}

model.eval()
with torch.no_grad():
    for i in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[i:i + BATCH_SIZE]
        batch_tensors = torch.stack([
            test_transform(Image.open(p).convert("RGB")) for p in batch_paths
        ]).to(device)
        logits = (model(batch_tensors) + model(batch_tensors.flip(-1))) / 2
        preds = logits.argmax(dim=1).cpu().tolist()
        for path, pred_idx in zip(batch_paths, preds):
            predictions_dict[str(int(path.stem))] = class_names[pred_idx]

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions_dict, f, indent=4, ensure_ascii=False)

print(f"{len(predictions_dict)} prédictions sauvegardées dans {output_json_path}")
